<a href="https://colab.research.google.com/github/Sopitas24/CPSN/blob/main/Ejercicios_de_histogramas_y_boxplots_javiera_cortes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

09 Visualización para ciencia de datos - Ejercicios de histogramas y boxplots (Core).txt
100%
# Visualización para ciencia de datos / Ejercicios de histogramas y boxplots (Core)

# HISTOGRAMAS Y BOXPLOTS: WORLD HAPPINESS REPORT
# Código completo para Google Colab.
#
# 1. Descarga el dataset:
#    https://www.kaggle.com/datasets/unsdsn/world-happiness
# 2. Descomprime el archivo.
# 3. Ejecuta este código y selecciona 2019.csv.
#
# IMPORTANTE:
# En este archivo, GDP per capita, Healthy life expectancy y Freedom
# representan contribuciones al puntaje de felicidad.
# No se interpretan como dólares, años de vida o porcentajes.
#
# Los valores atípicos se detectan con la regla de 1.5 × IQR.
# Un valor atípico no es necesariamente un error y no se elimina.

import io
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


# --------------------------------------------------
# 1. CARGAR LOS DATOS
# --------------------------------------------------

print("Selecciona el archivo 2019.csv.")
archivos = files.upload()

if len(archivos) != 1:
    raise ValueError("Selecciona un solo archivo CSV.")

nombre_archivo = list(archivos.keys())[0]

if not nombre_archivo.lower().endswith(".csv"):
    raise ValueError("El archivo debe tener extensión .csv.")

datos = pd.read_csv(io.BytesIO(archivos[nombre_archivo]))
datos.columns = datos.columns.str.strip()

variables = [
    "Score",
    "GDP per capita",
    "Healthy life expectancy",
    "Freedom to make life choices"
]

columnas_necesarias = ["Country or region"] + variables

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in datos.columns
]

if faltantes:
    raise ValueError(
        f"Faltan estas columnas: {', '.join(faltantes)}. "
        "Comprueba que seleccionaste 2019.csv."
    )


# --------------------------------------------------
# 2. EXPLORAR LOS DATOS
# --------------------------------------------------

print("\nPRIMERAS 10 FILAS")
display(datos.head(10))

print("\nÚLTIMAS 5 FILAS")
display(datos.tail(5))

print("\nINFORMACIÓN GENERAL")
datos.info()

print("\nCOLUMNAS DISPONIBLES")
print(datos.columns.tolist())

print("\nVALORES FALTANTES ORIGINALES")
print(datos[columnas_necesarias].isna().sum())


# --------------------------------------------------
# 3. PREPARAR LOS DATOS
# --------------------------------------------------

# Eliminar filas completamente duplicadas.
duplicados = datos.duplicated().sum()
datos = datos.drop_duplicates().copy()

print("\nFilas duplicadas eliminadas:", duplicados)

# Convertir las variables a números.
# Los valores inválidos se convierten en faltantes.
for variable in variables:
    datos[variable] = pd.to_numeric(datos[variable], errors="coerce")
    datos[variable] = datos[variable].replace(
        [float("inf"), float("-inf")],
        float("nan")
    )

print("\nESTADÍSTICAS DESCRIPTIVAS")
display(datos[variables].describe())

print("\nVALORES FALTANTES DESPUÉS DE LA CONVERSIÓN")
print(datos[variables].isna().sum())

# No rellenamos con promedios porque cambiaría las distribuciones.
# Cada gráfico utilizará los valores válidos de su variable.


# --------------------------------------------------
# 4. NOMBRES Y COLORES DE LOS GRÁFICOS
# --------------------------------------------------

nombres = {
    "Score": "Puntaje de felicidad",
    "GDP per capita": "Contribución del PIB per cápita",
    "Healthy life expectancy": "Contribución de la expectativa de vida saludable",
    "Freedom to make life choices": "Contribución de la libertad para decidir"
}

colores = {
    "Score": "skyblue",
    "GDP per capita": "lightgreen",
    "Healthy life expectancy": "salmon",
    "Freedom to make life choices": "plum"
}


# --------------------------------------------------
# 5. FUNCIÓN PARA GRAFICAR E INTERPRETAR
# --------------------------------------------------

def analizar_variable(datos, variable):
    # Seleccionar únicamente los valores válidos de esta variable.
    valores = datos[variable].dropna()

    print("\n" + "=" * 65)
    print(nombres[variable].upper())
    print("=" * 65)

    print("Valores válidos:", len(valores))
    print("Valores faltantes excluidos:", datos[variable].isna().sum())

    if valores.empty:
        print("No hay valores válidos para analizar.")
        return None

    # Estadísticas principales.
    media = valores.mean()
    mediana = valores.median()
    minimo = valores.min()
    maximo = valores.max()
    desviacion = valores.std()
    asimetria = valores.skew()

    # Q1 y Q3 delimitan el 50 % central de los datos.
    q1 = valores.quantile(0.25)
    q3 = valores.quantile(0.75)
    iqr = q3 - q1

    # Límites para detectar valores atípicos.
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    filtro_atipicos = (
        (datos[variable] < limite_inferior)
        | (datos[variable] > limite_superior)
    )

    atipicos = datos.loc[
        filtro_atipicos,
        ["Country or region", variable]
    ].sort_values(variable)

    atipicos_bajos = (valores < limite_inferior).sum()
    atipicos_altos = (valores > limite_superior).sum()

    # Crear una figura con histograma y boxplot para la misma variable.
    figura, ejes = plt.subplots(
        2,
        1,
        figsize=(10, 7),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]}
    )

    # Histograma: muestra la frecuencia en cada intervalo.
    frecuencias, limites, barras = ejes[0].hist(
        valores,
        bins=10,
        color=colores[variable],
        edgecolor="black",
        alpha=0.75
    )

    ejes[0].axvline(
        media,
        color="darkred",
        linestyle="--",
        linewidth=2,
        label=f"Media: {media:.3f}"
    )

    ejes[0].axvline(
        mediana,
        color="navy",
        linestyle=":",
        linewidth=2,
        label=f"Mediana: {mediana:.3f}"
    )

    ejes[0].set_title(
        f"Distribución de {nombres[variable].lower()} — 2019"
    )
    ejes[0].set_ylabel("Cantidad de países o regiones")
    ejes[0].legend()
    ejes[0].grid(axis="y", alpha=0.25)

    # Boxplot: muestra mediana, cuartiles y puntos atípicos.
    ejes[1].boxplot(
        valores,
        vert=False,
        whis=1.5,
        patch_artist=True,
        boxprops={
            "facecolor": colores[variable],
            "edgecolor": "black",
            "alpha": 0.75
        },
        medianprops={"color": "darkred", "linewidth": 2},
        whiskerprops={"color": "dimgray", "linestyle": "--"},
        capprops={"color": "dimgray"},
        flierprops={
            "marker": "o",
            "markerfacecolor": "orange",
            "markeredgecolor": "black",
            "markersize": 6
        }
    )

    ejes[1].set_yticks([])
    ejes[1].set_xlabel(nombres[variable])
    ejes[1].set_title("Boxplot: regla de 1.5 × IQR")
    ejes[1].grid(axis="x", alpha=0.25)

    plt.tight_layout()
    plt.show()

    # --------------------------------------------------
    # INTERPRETACIÓN BASADA EN LOS RESULTADOS
    # --------------------------------------------------

    print("\nESTADÍSTICAS")
    print(f"Media: {media:.3f}")
    print(f"Mediana: {mediana:.3f}")
    print(f"Mínimo: {minimo:.3f}")
    print(f"Máximo: {maximo:.3f}")
    print(f"Desviación estándar: {desviacion:.3f}")
    print(f"Q1: {q1:.3f}")
    print(f"Q3: {q3:.3f}")
    print(f"Rango intercuartil: {iqr:.3f}")

    print("\nINTERPRETACIÓN DE LA DISTRIBUCIÓN")
    print(
        f"El intervalo entre los cuartiles es {q1:.3f} a {q3:.3f}; "
        "delimita aproximadamente el 50 % central de los valores."
    )

    # Describir todos los intervalos empatados con mayor frecuencia.
    print("Intervalo o intervalos más frecuentes del histograma:")

    for i, frecuencia in enumerate(frecuencias):
        if frecuencia == frecuencias.max():
            print(
                f"- De {limites[i]:.3f} a {limites[i + 1]:.3f}: "
                f"{int(frecuencia)} países o regiones."
            )

    # Los umbrales son orientativos y complementan la lectura del gráfico.
    if pd.isna(asimetria):
        print("No se puede calcular la asimetría con estos valores.")
    else:
        print(f"Asimetría: {asimetria:.3f}.")

        if asimetria > 0.5:
            print("Se observa asimetría hacia valores altos, con cola a la derecha.")
        elif asimetria < -0.5:
            print("Se observa asimetría hacia valores bajos, con cola a la izquierda.")
        else:
            print("La asimetría es pequeña según el umbral orientativo de ±0.5.")
            print("Esto no demuestra que la distribución sea normal.")

    print("\nINTERPRETACIÓN DE LOS VALORES ATÍPICOS")
    print(f"Límite inferior: {limite_inferior:.3f}")
    print(f"Límite superior: {limite_superior:.3f}")
    print("Valores atípicos bajos:", atipicos_bajos)
    print("Valores atípicos altos:", atipicos_altos)

    if atipicos.empty:
        print("No se detectaron valores atípicos con la regla de 1.5 × IQR.")
    else:
        print("Países o regiones fuera de los límites:")
        display(atipicos)

        # Comparación descriptiva; no se modifica el dataset.
        sin_atipicos = valores[
            valores.between(limite_inferior, limite_superior)
        ]

        if not sin_atipicos.empty:
            print(f"Media con todos los valores: {media:.3f}")
            print(
                "Media excluyendo los atípicos, solo como comparación: "
                f"{sin_atipicos.mean():.3f}"
            )

        print(
            "Estos puntos pueden representar diferencias reales. "
            "Se conservan en el análisis."
        )

    return {
        "Variable": variable,
        "Valores_validos": len(valores),
        "Media": media,
        "Mediana": mediana,
        "Desviacion_estandar": desviacion,
        "Asimetria": asimetria,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Atipicos_bajos": atipicos_bajos,
        "Atipicos_altos": atipicos_altos
    }


# --------------------------------------------------
# 6. ANALIZAR LAS CUATRO VARIABLES
# --------------------------------------------------

resultados = []

for variable in variables:
    resultado = analizar_variable(datos, variable)

    if resultado is not None:
        resultados.append(resultado)


# --------------------------------------------------
# 7. RESUMEN FINAL
# --------------------------------------------------

resumen = pd.DataFrame(resultados)

print("\nRESUMEN DE LAS CUATRO VARIABLES")
display(resumen.round(3))

print("\nCONCLUSIONES GENERALES")
print("- Los histogramas muestran frecuencias, concentración y forma.")
print("- Los boxplots permiten comparar mediana, dispersión y puntos extremos.")
print("- La presencia de valores atípicos depende de los datos y del criterio usado.")
print("- No se debe afirmar que existen atípicos si el cálculo no los detecta.")
print("- Las contribuciones de PIB y salud no equivalen a dólares o años de vida.")
print("- Los resultados describen el archivo analizado de 2019.")


# --------------------------------------------------
# 8. GUARDAR EL RESUMEN
# --------------------------------------------------

resumen.to_csv(
    "resumen_felicidad_2019.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nResumen guardado en resumen_felicidad_2019.csv.")
print("Puedes descargarlo desde el panel de archivos de Google Colab.")
Mostrando 09 Visualización para ciencia de datos - Ejercicios de histogramas y boxplots (Core).txt